# Consumer & Classifier

Consumes events from Kafka, classifies them against MITRE ATT&CK tactics/techniques,
emits traces to Jaeger, and saves results to a local CSV file.

In [1]:
%pip install kafka-python opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 757.1 kB/s eta 0:00:00 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 1.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 2.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 809.6 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 2.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.8.0
    Uninstalling typing_extensions-4.8.0:
      Successfully uninstalled typing_extensions-4.8.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.24.3
    Uninstalling protobuf-4.24.3:
   

In [2]:
import json
import csv
import os
from kafka import KafkaConsumer
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

In [3]:
KAFKA_BROKER = "kafka:9092"
TOPIC = "events.raw"
GROUP_ID = "classifier-group"
OUTPUT_CSV = "/home/jovyan/data/classified_packets.csv"
JAEGER_ENDPOINT = "http://jaeger:4317"

MITRE_MAP = {
    "port_scan": {"tactic": "Reconnaissance", "technique": "Active Scanning", "technique_id": "T1595"},
    "brute_force": {"tactic": "Credential Access", "technique": "Brute Force", "technique_id": "T1110"},
    "data_exfiltration": {"tactic": "Exfiltration", "technique": "Exfiltration Over C2 Channel", "technique_id": "T1041"},
    "lateral_movement": {"tactic": "Lateral Movement", "technique": "Remote Services", "technique_id": "T1021"},
    "c2_communication": {"tactic": "Command and Control", "technique": "Application Layer Protocol", "technique_id": "T1071"},
    "privilege_escalation": {"tactic": "Privilege Escalation", "technique": "Exploitation for Privilege Escalation", "technique_id": "T1068"},
    "normal_traffic": {"tactic": "N/A", "technique": "N/A", "technique_id": "N/A"},
}

In [4]:
provider = TracerProvider()
exporter = OTLPSpanExporter(endpoint=JAEGER_ENDPOINT, insecure=True)
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("classifier")

In [5]:
def classify(event):
    event_type = event.get("event_type", "normal_traffic")
    mapping = MITRE_MAP.get(event_type, MITRE_MAP["normal_traffic"])
    return {
        **event,
        "mitre_tactic": mapping["tactic"],
        "mitre_technique": mapping["technique"],
        "mitre_technique_id": mapping["technique_id"],
        "confidence": 0.95 if event_type != "normal_traffic" else 1.0,
    }

In [6]:
fieldnames = [
    "event_id", "timestamp", "src_ip", "dst_ip", "src_port", "dst_port",
    "protocol", "payload_size", "event_type",
    "mitre_tactic", "mitre_technique", "mitre_technique_id", "confidence",
]

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    group_id=GROUP_ID,
    auto_offset_reset="earliest",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=10000,
)

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for msg in consumer:
        with tracer.start_as_current_span("kafka.consume") as span:
            event = msg.value
            span.set_attribute("event.id", event["event_id"])

            with tracer.start_as_current_span("classification"):
                result = classify(event)

            with tracer.start_as_current_span("storage.write"):
                writer.writerow(result)

            print(f"Classified: {result['event_id']} -> {result['mitre_tactic']} / {result['mitre_technique_id']}")

consumer.close()
print(f"\nDone. Results saved to {OUTPUT_CSV}.")

Classified: 2e7f9b77-c1e6-4c2e-8c7d-151bcfe60e0b -> Credential Access / T1110
Classified: 949bd870-f195-4c6d-be29-d3a527ca93fb -> N/A / N/A
Classified: 8e88840a-8ff4-4bde-9d7d-93b54ae97103 -> Privilege Escalation / T1068
Classified: 92f5c0e0-faf2-4fed-b6b9-0dbba3f9445c -> Privilege Escalation / T1068
Classified: fd358c83-8f88-4a43-80a1-d254e63eced4 -> Command and Control / T1071
Classified: adff4330-9b03-4260-8394-eb584216bb14 -> Reconnaissance / T1595
Classified: 3773c6db-6112-4937-af50-62f46c9f7e60 -> Privilege Escalation / T1068
Classified: 5155e795-2f2d-4cd0-9379-d803d71b25fd -> Reconnaissance / T1595
Classified: 81907db2-37e6-46eb-8a1a-28382fb6bcf0 -> Exfiltration / T1041
Classified: 75c7c81c-e710-4c91-a0ef-c3ceff34fa4e -> Command and Control / T1071
Classified: 7e57d226-545f-43e6-85cb-07d405aa3c07 -> Credential Access / T1110
Classified: af8e4d85-125a-4a58-924c-fcbb79b82df3 -> Lateral Movement / T1021
Classified: f855baac-393d-42ca-b01f-86536081fdf0 -> Reconnaissance / T1595
Clas